# 02. 외부 데이터 EDA — 157개 목표 bbox 선별 및 결측 점검

**기본 환경 설정 → 전체 외부 데이터 확인 → 대회 보정 v1 분포 → 157개 목표 bbox 선별 → 선택 데이터 결측·겹침 검사** 순서로 실행합니다.

- 입력은 `AIHUB_EXTRACT_DIR` 또는 기본 `data/external/aihub_combo`이며, 받은 전체 폴더를 확인합니다. 다운로드·압축 해제는 실행하지 않습니다.
- 클래스 키는 `drug_N`을 숫자로 바꾼 약 코드입니다. 대회 보정 v1과 공통인 클래스만 157개 목표로 보충합니다.
- 사용하는 결과는 §7의 `selected_pills`입니다. 현재 데이터에서 선택 bbox 7,803개와 원본 이미지 5,499장이며, 선택 bbox 수와 원본 이미지의 전체 bbox 수는 구분합니다.
- bbox 결측·좌표 오류·중복·겹침은 §8~9에서 확인합니다. 파일명 기준 누락과 픽셀 속 실제 미라벨 객체는 다를 수 있습니다.
- 원본 이미지·JSON·보정 기록은 수정하지 않습니다. 이 노트북은 후보·검사표를 만들며 crop·학습 데이터 병합·YOLO 변환은 수행하지 않습니다.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("pill-bbox 저장소에서 실행하세요. Colab은 저장소를 먼저 준비하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
SEED = 42
EXCLUDE = []  # 현재 받은 전체 폴더를 확인한다. 필요할 때만 TL_2 등의 구분자를 넣는다.
extract_env = os.getenv("AIHUB_EXTRACT_DIR")
EXTRACT_DIR = Path(extract_env).expanduser() if extract_env else ROOT / "data/external/aihub_combo"
if not EXTRACT_DIR.is_absolute():
    EXTRACT_DIR = ROOT / EXTRACT_DIR
print("외부 데이터 폴더:", EXTRACT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from data_process.aihub_data import list_combo_images, load_combo_annotations, zip_key
from data_process.data_processing import build_cat_df, build_img_df, list_images, load_annotations
from visualization import draw_bboxes, image_grid, set_korean_font, show_image

pd.set_option("display.max_columns", 40)
print("한글 폰트:", set_korean_font())

## 1. 전체 파일 현황

이미지·JSON 수를 폴더별로 집계합니다. 이미지 전체를 메모리에 읽지는 않습니다.
같은 파일 이름이 여러 폴더에 있으면 자동으로 하나를 선택하지 않고 중단해 중복 목록부터 확인합니다.


In [ ]:
if not EXTRACT_DIR.is_dir():
    raise FileNotFoundError(f"압축을 푼 외부 데이터 폴더가 없습니다: {EXTRACT_DIR}")
extracted_dirs = sorted(
    folder
    for folder in EXTRACT_DIR.iterdir()
    if folder.is_dir() and not folder.name.startswith(".") and zip_key(folder) not in EXCLUDE
)
if not extracted_dirs:
    raise FileNotFoundError("분석할 하위 폴더가 없습니다. EXTRACT_DIR과 EXCLUDE를 확인하세요.")
image_index = list_combo_images(extracted_dirs)
folder_inventory = pd.DataFrame(
    [
        {
            "폴더": folder.name,
            "이미지 수": int(image_index["zip_name"].eq(folder.name).sum()),
            "JSON 수": sum(1 for _ in folder.rglob("*.json")),
        }
        for folder in extracted_dirs
    ]
)
display(folder_inventory)
print(f"전체 이미지 파일: {len(image_index):,}장")
print(f"전체 JSON: {folder_inventory['JSON 수'].sum():,}개")
if image_index.empty:
    raise FileNotFoundError("이미지가 없습니다. TS/VS 원천 이미지 폴더를 확인하세요.")
duplicate_images = image_index[image_index["file_name"].duplicated(keep=False)]
print(f"동일 이름 중복 파일: {len(duplicate_images):,}개")
if not duplicate_images.empty:
    display(duplicate_images)
    raise ValueError("이미지 이름 중복을 먼저 확인해야 안전하게 라벨을 연결할 수 있습니다.")

## 2. 라벨 연결과 이미지 전체 요약

기존 로더로 전체 JSON 메타데이터를 읽습니다. 이미지가 없는 라벨도 버리지 않고 별도로 집계합니다.
`ann_df`는 모든 외부 라벨, `available_ann_df`는 실제 이미지 파일과 연결되는 라벨입니다.
클래스 비교에는 두 범위를 함께 표시하여 **라벨만 있고 이미지가 없는 클래스**를 구분합니다.


In [ ]:
ann_df, load_error_df = load_combo_annotations(extracted_dirs)
if ann_df.empty:
    display(load_error_df)
    raise ValueError("읽을 수 있는 외부 라벨이 없습니다. TL/VL 라벨 폴더를 확인하세요.")
invalid_class_rows = ann_df[ann_df["category_id"].isna()]
print(f"약 코드 해석 불가 bbox: {len(invalid_class_rows):,}개")
if not invalid_class_rows.empty:
    display(invalid_class_rows[["source_json", "drug_N", "dl_name"]].head(20))
    raise ValueError("해석할 수 없는 약 코드가 있어 클래스 비교를 중단합니다.")
image_names = set(image_index["file_name"])
missing_image_labels = ann_df[~ann_df["file_name"].isin(image_names)]
available_ann_df = ann_df[ann_df["file_name"].isin(image_names)].copy()
img_df = build_img_df(ann_df, list(image_index["path"])).merge(
    image_index[["file_name", "zip_name", "path"]], on="file_name", validate="one_to_one"
)
cat_df = build_cat_df(ann_df)
img_path = dict(zip(image_index["file_name"], image_index["path"], strict=True))
summary = pd.DataFrame(
    [
        ("디스크 전체 이미지", len(image_index)),
        ("라벨이 연결된 이미지", int(img_df["n_boxes"].gt(0).sum())),
        ("라벨 없는 이미지", int(img_df["n_boxes"].eq(0).sum())),
        ("전체 bbox 라벨", len(ann_df)),
        ("이미지와 연결된 bbox", len(available_ann_df)),
        ("라벨에만 있는 이미지 이름", missing_image_labels["file_name"].nunique()),
        ("전체 라벨 클래스", ann_df["category_id"].nunique()),
        ("이미지가 있는 클래스", available_ann_df["category_id"].nunique()),
        ("JSON 로드 오류·주의 기록", len(load_error_df)),
    ],
    columns=["항목", "개수"],
)
display(summary)
if not load_error_df.empty:
    display(load_error_df.head(20))
if not missing_image_labels.empty:
    print("이미지가 없는 라벨 예시 (자동 삭제하지 않음)")
    display(missing_image_labels[["file_name", "drug_N", "dl_name", "zip_name"]].head(20))

display(img_df.groupby("zip_name").agg(이미지=("file_name", "size"), bbox=("n_boxes", "sum")))
ax = img_df["zip_name"].value_counts().sort_index().plot.bar(figsize=(11, 4), rot=30)
ax.set(title="다운로드한 외부 이미지 전체 현황", xlabel="원천 폴더", ylabel="이미지 수")
plt.tight_layout()
plt.show()

## 3. 전체 범위의 이미지 예시

각 원천 폴더에서 최대 2장씩 seed=42로 뽑아 표시합니다. 원본 이미지 전체의 무결성 검사나 누락 판정은 아닙니다.
라벨이 있으면 기존 bbox와 약 이름을 함께 표시합니다.


In [ ]:
sample = img_df.groupby("zip_name", group_keys=False).sample(n=1, random_state=SEED)
# 각 폴더의 두 번째 예시가 가능하면 추가한다.
remaining = img_df.drop(sample.index)
if not remaining.empty:
    sample = pd.concat(
        [sample, remaining.groupby("zip_name", group_keys=False).sample(n=1, random_state=SEED)]
    ).sort_values(["zip_name", "file_name"])
fig, axes = image_grid(len(sample), cols=4)
for ax, row in zip(axes, sample.itertuples(), strict=True):
    show_image(
        ax, row.path, title=f"{row.zip_name} / bbox {row.n_boxes}개\n{row.file_name}", fontsize=6
    )
    draw_bboxes(ax, ann_df[ann_df["file_name"].eq(row.file_name)], label_col="dl_name")
fig.tight_layout()
plt.show()

## 4. 대회 train과 클래스 비교 — 공통 / 외부에만 / 대회에만

대회 원본 train 라벨의 `category_id`와 외부 라벨의 `drug_N` 숫자 변환 값을 비교합니다.
**외부에만 있는 클래스**는 대회 train 라벨에 없는 약 코드이며, 대회 test에도 없다는 뜻은 아닙니다.
전체 외부 라벨 기준 집합과 이미지가 실제 있는 집합을 모두 집계합니다. 이름 차이는 별도 참고값으로 표시하며 자동으로 합치거나 고치지 않습니다.


In [ ]:
raw_dir = ROOT / "data/raw"
if (raw_dir / "sprint_ai_project1_data").is_dir():
    raw_dir = raw_dir / "sprint_ai_project1_data"
if not (raw_dir / "train_annotations").is_dir():
    raise FileNotFoundError(f"대회 train_annotations가 없습니다: {raw_dir}")
comp_ann_df, comp_error_df = load_annotations(raw_dir / "train_annotations")
if comp_ann_df.empty or not comp_error_df.empty or comp_ann_df["category_id"].isna().any():
    display(comp_error_df)
    raise ValueError(
        "대회 라벨 로드 결과를 확인하세요. 불완전한 클래스 집합으로 비교하지 않습니다."
    )
comp_cat_df = build_cat_df(comp_ann_df)
comp_classes = set(comp_cat_df["category_id"].astype(int))
external_classes = set(cat_df["category_id"].astype(int))
available_classes = set(available_ann_df["category_id"].astype(int))
class_summary = pd.DataFrame(
    [
        {
            "비교 범위": scope,
            "대회 클래스": len(comp_classes),
            "외부 클래스": len(classes),
            "공통": len(classes & comp_classes),
            "외부에만": len(classes - comp_classes),
            "대회에만": len(comp_classes - classes),
        }
        for scope, classes in [
            ("전체 외부 라벨", external_classes),
            ("이미지가 있는 외부 라벨", available_classes),
        ]
    ]
)
display(class_summary)

available_counts = available_ann_df.groupby("category_id").agg(
    available_boxes=("annotation_id", "size"), available_images=("file_name", "nunique")
)
class_comparison = cat_df.merge(available_counts, on="category_id", how="left").merge(
    comp_cat_df[["category_id", "dl_name"]].rename(columns={"dl_name": "competition_name"}),
    on="category_id",
    how="left",
    validate="one_to_one",
)
class_comparison[["available_boxes", "available_images"]] = (
    class_comparison[["available_boxes", "available_images"]].fillna(0).astype(int)
)
class_comparison["구분"] = class_comparison["category_id"].map(
    lambda category: "공통" if category in comp_classes else "외부에만"
)
external_only_df = class_comparison[class_comparison["구분"].eq("외부에만")].copy()
shared_class_df = class_comparison[class_comparison["구분"].eq("공통")].copy()
competition_only_df = comp_cat_df[~comp_cat_df["category_id"].isin(external_classes)].copy()
print("대회 전용·외부 전용 클래스는 보충 대상에서 제외합니다.")
columns = ["category_id", "drug_N", "dl_name", "n_boxes", "n_images", "available_images"]
print("공통 클래스 (같은 약 코드의 이름 표기 비교)")
with pd.option_context("display.max_rows", None):
    display(shared_class_df[columns + ["competition_name"]].reset_index(drop=True))

## 5. 01_eda 보정 v1 기준 분포와 157개 보충 목표

기존 파이프라인이 만든 보정 v1 라벨과 fingerprint를 확인합니다. 목표는 사용자가 선택한 **클래스당 157 bbox**입니다. 외부에 없는 대회 클래스는 외부 보충 대상에서 제외합니다.


In [ ]:
import json

import numpy as np

from data_process.data_processing import bbox_flags
from data_process.make_labels import check_labels, file_sha256

label_path = ROOT / "data/processed/labels/labels_v1.csv"
label_meta_path = label_path.with_suffix(".json")
if not label_path.is_file() or not label_meta_path.is_file():
    raise FileNotFoundError(
        "보정 v1 라벨이 필요합니다. 기존 data_process.data_main --label v1을 먼저 실행하세요."
    )
label_meta = json.loads(label_meta_path.read_text())
if file_sha256(label_path) != label_meta["labels_sha256"]:
    raise ValueError("v1 라벨과 메타데이터 fingerprint가 다릅니다. 파이프라인을 다시 실행하세요.")
comp_corrected_df = pd.read_csv(label_path)
label_checks = check_labels(comp_corrected_df)
if any(
    label_checks[key] for key in ["bbox_missing", "out_of_image", "non_positive", "duplicate_label"]
):
    raise ValueError(f"보정 v1 라벨 검사를 통과하지 못했습니다: {label_checks}")
if set(comp_corrected_df["file_name"]) != set(comp_ann_df["file_name"]):
    raise ValueError("현재 대회 원본과 v1의 이미지 구성이 다릅니다. v1을 다시 생성하세요.")
raw_counts = comp_ann_df.groupby("category_id").size()
corrected_counts = comp_corrected_df.groupby("category_id").size()
common_classes = sorted(set(corrected_counts.index) & available_classes)
TARGET_BBOX = 157  # 이번에 사용하기로 확정한 클래스별 보충 목표
class_balance = (
    comp_cat_df[["category_id", "drug_N", "dl_name"]]
    .set_index("category_id")
    .assign(raw_bbox=raw_counts, competition_bbox=corrected_counts)
)
class_balance["common_class"] = class_balance.index.isin(common_classes)
class_balance["target_bbox"] = TARGET_BBOX
class_balance["needed_bbox"] = (TARGET_BBOX - class_balance["competition_bbox"]).clip(lower=0)
class_balance.loc[~class_balance["common_class"], "needed_bbox"] = 0
class_balance["group"] = np.select(
    [class_balance["competition_bbox"].le(5), class_balance["competition_bbox"].le(20)],
    ["희귀 (≤5)", "소수 (6–20)"],
    default="다수 (≥21)",
)
print(f"보정 v1: {len(comp_corrected_df):,} bbox / {len(corrected_counts)}종")
print(f"공통 {len(common_classes)}종의 목표: 클래스당 {TARGET_BBOX:,} bbox")
with pd.option_context("display.max_rows", None):
    display(class_balance.sort_values("competition_bbox"))
plot_counts = class_balance.sort_values("competition_bbox")
ax = plot_counts["competition_bbox"].plot.barh(figsize=(11, 14))
ax.axvline(TARGET_BBOX, color="red", linestyle="--", label=f"보충 목표 {TARGET_BBOX}")
ax.set(title="대회 보정 v1 클래스별 bbox 분포", xlabel="bbox 수", ylabel="약 코드")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 개별 bbox 선별에 필요한 후보와 기본 검사

공통 클래스의 bbox 후보를 모읍니다. 좌표 결측·이미지 경계 초과·크기 오류·같은 약 중복 라벨이 있는 이미지와 대회와 동일 파일명은 선별에서 제외합니다. 라벨이 아예 없는 약은 이 검사만으로 제외되지 않으므로 선택 후 §8에서 따로 점검합니다.


In [ ]:
common_bbox_candidates = available_ann_df[
    available_ann_df["category_id"].isin(common_classes)
].merge(image_index[["file_name", "path"]], on="file_name", validate="many_to_one")

box_flags = bbox_flags(available_ann_df)
numeric = available_ann_df[["x", "y", "w", "h", "width", "height"]].apply(
    pd.to_numeric, errors="coerce"
)
invalid_box = (
    ~np.isfinite(numeric).all(axis=1)
    | numeric[["width", "height"]].le(0).any(axis=1)
    | box_flags["out_of_image"]
    | box_flags["non_positive"]
    | available_ann_df.duplicated(["file_name", "category_id"], keep=False)
)
invalid_files = set(available_ann_df.loc[invalid_box, "file_name"])
comp_image_names = set()
for split in ["train", "test"]:
    image_dir = raw_dir / f"{split}_images"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"대회 {split} 이미지 폴더가 필요합니다: {image_dir}")
    comp_image_names.update(path.name for path in list_images(image_dir))
comp_combos = set(comp_corrected_df["file_name"].str.split("_").str[0])
comp_combos.update(name.split("_")[0] for name in comp_image_names if name.startswith("K-"))

print(f"공통 클래스 개별 bbox 후보: {len(common_bbox_candidates):,}개")
print(f"좌표·중복 오류로 제외할 이미지: {len(invalid_files):,}장")

## 7. 사용할 데이터 — 클래스당 157개 목표의 개별 bbox 선택

기존과 같은 seed 42로 클래스별 부족량만큼 비복원 추출합니다. `selected_pills`는 **선택한 개별 알약 bbox 목록**입니다. 이 bbox가 들어 있는 원본 이미지를 통째로 사용하면 선택하지 않은 대회 알약과 외부 전용 알약도 함께 들어옵니다. 후보 bbox가 부족한 클래스는 157개 미만으로 남습니다.


In [ ]:
pill_candidate_pool = common_bbox_candidates[
    ~common_bbox_candidates["file_name"].isin(invalid_files | comp_image_names)
].copy()
pill_candidate_pool["combo"] = pill_candidate_pool["file_name"].str.split("_").str[0]
pill_candidate_pool["shares_competition_combo"] = pill_candidate_pool["combo"].isin(comp_combos)
pill_candidate_pool = pill_candidate_pool.drop_duplicates(
    ["file_name", "category_id", "x", "y", "w", "h"]
)
# 같은 클래스의 서로 다른 이미지를 seed를 고정하여 비복원 추출한다.
shuffled_pills = pill_candidate_pool.sample(frac=1, random_state=SEED)
pill_order = shuffled_pills.groupby("category_id").cumcount()
pill_need = shuffled_pills["category_id"].map(class_balance["needed_bbox"])
selected_pills = (
    shuffled_pills[pill_order.lt(pill_need)]
    .sort_values(["category_id", "file_name"])
    .reset_index(drop=True)
)
pill_balance_report = class_balance.loc[common_classes].copy()
pill_balance_report["candidate_bbox"] = (
    pill_candidate_pool.groupby("category_id").size().reindex(common_classes, fill_value=0)
)
pill_balance_report["selected_bbox"] = (
    selected_pills.groupby("category_id").size().reindex(common_classes, fill_value=0)
)
pill_balance_report["after_bbox"] = (
    pill_balance_report["competition_bbox"] + pill_balance_report["selected_bbox"]
)
pill_balance_report["remaining_bbox"] = (TARGET_BBOX - pill_balance_report["after_bbox"]).clip(
    lower=0
)
pill_balance_report["status"] = np.where(
    pill_balance_report["remaining_bbox"].eq(0), "목표 충족", "후보 부족"
)
assert set(selected_pills["category_id"]).issubset(common_classes)
assert not selected_pills.duplicated(["file_name", "category_id"]).any()
assert pill_balance_report["after_bbox"].le(TARGET_BBOX).all()
n_pill_images = selected_pills["file_name"].nunique()
print(f"수집한 공통 알약: {len(selected_pills):,} bbox / {n_pill_images:,}개 이미지")
n_pill_met = int(pill_balance_report["remaining_bbox"].eq(0).sum())
print(f"목표 {TARGET_BBOX}개 충족: {n_pill_met}/{len(common_classes)}종")
with pd.option_context("display.max_rows", None):
    display(
        pill_balance_report[
            [
                "dl_name",
                "competition_bbox",
                "needed_bbox",
                "candidate_bbox",
                "selected_bbox",
                "after_bbox",
                "remaining_bbox",
                "status",
            ]
        ]
    )
display(
    selected_pills[
        [
            "category_id",
            "dl_name",
            "file_name",
            "path",
            "x",
            "y",
            "w",
            "h",
            "source_json",
            "shares_competition_combo",
        ]
    ].head(20)
)
pill_plot = pill_balance_report.sort_values("competition_bbox")
ax = pill_plot[["competition_bbox", "selected_bbox"]].plot.barh(
    stacked=True, figsize=(12, 14), color=["steelblue", "darkorange"]
)
ax.axvline(TARGET_BBOX, color="red", linestyle="--", label=f"목표 {TARGET_BBOX}")
ax.set(title="공통 알약 클래스별 보충 후보 수집 결과", xlabel="bbox 수", ylabel="약 코드")
ax.legend()
plt.tight_layout()
plt.show()

### 7.1 클래스별 외부 데이터 반입 수량 표

현재 데이터와 선별 결과로 수량을 자동 계산합니다. 공통 클래스만 표시하고 마지막에 합계를 붙입니다.


In [ ]:
supplement_counts = (
    pill_balance_report[
        ["competition_bbox", "needed_bbox", "selected_bbox", "after_bbox", "remaining_bbox"]
    ]
    .rename_axis("클래스 번호")
    .rename(
        columns={
            "competition_bbox": "대회 현재",
            "needed_bbox": "추가 필요",
            "selected_bbox": "선택 bbox",
            "after_bbox": "선택 후",
            "remaining_bbox": "남은 부족",
        }
    )
    .astype(int)
    .reset_index()
)
print(f"공통 클래스별 목표 {TARGET_BBOX}개 / 단위: 알약 bbox")
with pd.option_context("display.max_rows", None):
    display(supplement_counts.style.hide(axis="index"))

## 8. 선택한 157개 목표 데이터의 결측·겹침 검사

범위는 §7의 **선택 bbox**와 **그 bbox가 포함된 원본 이미지 전체**입니다. 전체 외부 데이터나 다른 선별안은 섞지 않습니다.

1. 선택 bbox의 라벨·좌표 결측, 크기·경계 오류, 같은 약 중복 라벨, 이미지 파일 존재를 확인합니다.
2. 원본 이미지 파일명에 있는 약 코드와 실제 라벨을 비교해 **대회 알약 / 외부 전용 알약의 bbox 누락**을 나눠 셉니다.
3. 양의 IoU가 있는 겹침 쌍, 선택 bbox가 포함된 쌍, 완전 동일 좌표 쌍을 확인합니다. 겹침은 라벨 오류 확정이 아니라 사람 검수 대상입니다.

선택 bbox는 앞 단계에서 라벨·좌표가 있는 후보로 골랐으므로 그 자체의 결측이 0이어도 원본 이미지에 함께 찍힌 다른 약의 bbox는 없을 수 있습니다. 파일명 기준 알약 수는 실제 픽셀을 인식해 센 수가 아닙니다.

요약·클래스별 개수·이미지별 점검표·누락 목록·겹침 쌍은 `runs/original157_missing_review/`에 CSV로 저장합니다. 원본 라벨을 보정하거나 이미지를 제외·삭제하지 않습니다.


In [ ]:
from data_process.data_processing import find_box_pairs

AUDIT_PREVIEW_ROWS = 30
if any(
    name not in globals()
    for name in ["selected_pills", "available_ann_df", "image_index", "comp_corrected_df"]
):
    raise RuntimeError("먼저 7절의 157개 목표 bbox 선별까지 실행하세요.")
audit_selected = selected_pills.copy()
audit_files = sorted(audit_selected["file_name"].unique())
audit_competition_ids = set(comp_corrected_df["category_id"])
audit_image_lookup = (
    image_index[image_index["file_name"].isin(audit_files)][["file_name", "path"]]
    .sort_values("file_name")
    .copy()
)
audit_image_lookup.insert(0, "이미지 번호", np.arange(1, len(audit_image_lookup) + 1))
audit_numbers = audit_image_lookup.set_index("file_name")["이미지 번호"]
audit_annotations = available_ann_df[available_ann_df["file_name"].isin(audit_files)].copy()
audit_selected_keys = set(
    zip(audit_selected["source_json"], audit_selected["annotation_id"], strict=True)
)
audit_annotations["선택 bbox"] = [
    (source, annotation) in audit_selected_keys
    for source, annotation in zip(
        audit_annotations["source_json"], audit_annotations["annotation_id"], strict=True
    )
]
audit_annotations["대회 클래스"] = audit_annotations["category_id"].isin(audit_competition_ids)
# 파일명의 코드와 실제 라벨의 코드 집합을 비교한다. 픽셀에서 알약을 인식한 결과는 아니다.
audit_expected = audit_image_lookup[["file_name"]].copy()
audit_expected["category_id"] = audit_expected["file_name"].map(
    lambda name: sorted({int(code) for code in name.split("_")[0].split("-")[1:]})
)
audit_expected = audit_expected.explode("category_id")
audit_expected["category_id"] = audit_expected["category_id"].astype(int)
audit_presence = audit_expected.merge(
    audit_annotations[["file_name", "category_id"]].drop_duplicates(),
    on=["file_name", "category_id"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)
audit_missing = audit_presence[audit_presence["_merge"].eq("left_only")][
    ["file_name", "category_id"]
].copy()
audit_extra = audit_presence[audit_presence["_merge"].eq("right_only")][
    ["file_name", "category_id"]
].copy()
for frame in [audit_missing, audit_extra]:
    frame["대회 클래스"] = frame["category_id"].isin(audit_competition_ids)
    frame["이미지 번호"] = frame["file_name"].map(audit_numbers)

# 라벨이 있다는 것과 좌표·크기가 유효하다는 것을 구분한다.
audit_numeric_columns = ["x", "y", "w", "h", "width", "height"]
audit_numeric = audit_annotations[audit_numeric_columns].apply(pd.to_numeric, errors="coerce")
audit_finite = np.isfinite(audit_numeric).all(axis=1)
audit_flags = bbox_flags(audit_annotations)
audit_annotations["좌표 결측·비정상"] = ~audit_finite
audit_annotations["너비·높이 0 이하"] = (
    audit_numeric[["w", "h", "width", "height"]].le(0).any(axis=1)
)
audit_annotations["이미지 경계 초과"] = audit_flags["out_of_image"]
audit_annotations["같은 약 중복 라벨"] = audit_annotations.duplicated(
    ["file_name", "category_id"], keep=False
)
audit_error_columns = [
    "좌표 결측·비정상",
    "너비·높이 0 이하",
    "이미지 경계 초과",
    "같은 약 중복 라벨",
]
audit_invalid = audit_annotations[audit_annotations[audit_error_columns].any(axis=1)].copy()
audit_image_lookup["파일 존재"] = audit_image_lookup["path"].map(lambda path: Path(path).is_file())

# 양의 면적 겹침은 검수 후보다. 실제 알약끼리 맞닿거나 가려진 경우도 있어 오류로 확정하지 않는다.
audit_valid_boxes = audit_annotations[audit_finite & ~audit_annotations["너비·높이 0 이하"]].copy()
audit_pairs = find_box_pairs(audit_valid_boxes, max_gap=0)
audit_overlaps = audit_pairs[audit_pairs["iou"].gt(0)].copy()
audit_overlaps["선택 bbox 포함"] = [
    (row.source_json_1, row.annotation_id_1) in audit_selected_keys
    or (row.source_json_2, row.annotation_id_2) in audit_selected_keys
    for row in audit_overlaps.itertuples()
]
audit_overlaps["대회 bbox 포함"] = audit_overlaps["category_id_1"].isin(
    audit_competition_ids
) | audit_overlaps["category_id_2"].isin(audit_competition_ids)
audit_overlaps["대회 bbox끼리"] = audit_overlaps["category_id_1"].isin(
    audit_competition_ids
) & audit_overlaps["category_id_2"].isin(audit_competition_ids)
audit_overlaps["완전 동일 좌표"] = np.isclose(audit_overlaps["iou"], 1.0, rtol=0, atol=1e-9)
audit_overlaps["이미지 번호"] = audit_overlaps["file_name"].map(audit_numbers)
audit_overlaps = audit_overlaps.sort_values(["선택 bbox 포함", "iou"], ascending=[False, False])
audit_selected_presence = (
    audit_selected[["file_name", "category_id"]]
    .drop_duplicates()
    .merge(
        available_ann_df[["file_name", "category_id"]].drop_duplicates(),
        on=["file_name", "category_id"],
        how="left",
        indicator=True,
        validate="one_to_one",
    )
)
audit_summary = pd.DataFrame(
    [
        ("선택한 알약 bbox", len(audit_selected)),
        ("해당 원본 이미지", len(audit_image_lookup)),
        ("선택 bbox 라벨 없음", int(audit_selected_presence["_merge"].eq("left_only").sum())),
        ("선택 bbox 좌표·중복 오류", int(audit_invalid["선택 bbox"].sum())),
        ("선택 이미지 파일 없음", int((~audit_image_lookup["파일 존재"]).sum())),
        ("원본 이미지의 대회 알약 bbox 누락", int(audit_missing["대회 클래스"].sum())),
        ("원본 이미지의 외부 전용 알약 bbox 누락", int((~audit_missing["대회 클래스"]).sum())),
        ("원본 이미지 전체 bbox 겹침 쌍", len(audit_overlaps)),
        ("그중 선택 bbox가 포함된 겹침 쌍", int(audit_overlaps["선택 bbox 포함"].sum())),
        ("그중 대회 bbox끼리 겹침 쌍", int(audit_overlaps["대회 bbox끼리"].sum())),
        ("그중 완전 동일 좌표 겹침 쌍", int(audit_overlaps["완전 동일 좌표"].sum())),
        ("원본 이미지 좌표·중복 오류 bbox", len(audit_invalid)),
        ("파일명에 없는 클래스 라벨", len(audit_extra)),
    ],
    columns=["검사 항목", "개수"],
)

# 이미지별 검사 목록. 겹침 수는 알약 수가 아닌 쌍 수다.
audit_image_report = audit_image_lookup.copy()
for column, counts in {
    "선택 bbox 수": audit_selected.groupby("file_name").size(),
    "전체 bbox 수": audit_annotations.groupby("file_name").size(),
    "누락 bbox 수": audit_missing.groupby("file_name").size(),
    "대회 알약 누락 수": audit_missing[audit_missing["대회 클래스"]].groupby("file_name").size(),
    "겹침 쌍 수": audit_overlaps.groupby("file_name").size(),
    "선택 bbox 포함 겹침 쌍": audit_overlaps[audit_overlaps["선택 bbox 포함"]]
    .groupby("file_name")
    .size(),
    "좌표·중복 오류 수": audit_invalid.groupby("file_name").size(),
}.items():
    audit_image_report[column] = audit_image_report["file_name"].map(counts).fillna(0).astype(int)
audit_image_report["파일명 기준 알약 수"] = audit_image_report["file_name"].map(
    lambda name: len(name.split("_")[0].split("-")[1:])
)
audit_problem_images = audit_image_report[
    audit_image_report[["누락 bbox 수", "겹침 쌍 수", "좌표·중복 오류 수"]].gt(0).any(axis=1)
    | ~audit_image_report["파일 존재"]
].copy()

# 클래스별 간단 표: 같은 클래스가 같은 쌍 양쪽에 있어도 관여 횟수는 한 번 센다.
audit_overlap_class_rows = pd.concat(
    [
        audit_overlaps[
            [
                "file_name",
                "source_json_1",
                "annotation_id_1",
                "source_json_2",
                "annotation_id_2",
                "category_id_1",
                "선택 bbox 포함",
            ]
        ].rename(columns={"category_id_1": "category_id"}),
        audit_overlaps[
            [
                "file_name",
                "source_json_1",
                "annotation_id_1",
                "source_json_2",
                "annotation_id_2",
                "category_id_2",
                "선택 bbox 포함",
            ]
        ].rename(columns={"category_id_2": "category_id"}),
    ]
).drop_duplicates()
audit_class_report = pd.DataFrame(index=pd.Index(sorted(audit_competition_ids), name="클래스 번호"))
audit_class_report["선택 bbox 수"] = (
    audit_selected.groupby("category_id").size().reindex(audit_class_report.index, fill_value=0)
)
audit_class_report["원본 이미지 누락 수"] = (
    audit_missing.groupby("category_id").size().reindex(audit_class_report.index, fill_value=0)
)
audit_class_report["선택 bbox 포함 겹침 관여 쌍"] = (
    audit_overlap_class_rows[audit_overlap_class_rows["선택 bbox 포함"]]
    .groupby("category_id")
    .size()
    .reindex(audit_class_report.index, fill_value=0)
)
audit_class_report["좌표·중복 오류 수"] = (
    audit_invalid.groupby("category_id").size().reindex(audit_class_report.index, fill_value=0)
)
audit_class_report = audit_class_report.astype(int)
audit_output_dir = ROOT / "runs/original157_missing_review"
audit_output_dir.mkdir(parents=True, exist_ok=True)
for name, frame in {
    "summary": audit_summary,
    "class_report": audit_class_report,
    "image_report": audit_image_report,
    "problem_images": audit_problem_images,
    "missing_bbox": audit_missing,
    "overlap_pairs": audit_overlaps,
    "invalid_bbox": audit_invalid,
    "extra_class_labels": audit_extra,
}.items():
    frame.to_csv(
        audit_output_dir / f"{name}.csv", index=name == "class_report", encoding="utf-8-sig"
    )
print("검사 범위: 157개 목표로 선택한 bbox와 해당 원본 이미지만 검사합니다.")
display(audit_summary.style.hide(axis="index"))
print("대회 클래스별 결측·겹침 확인 (문제가 있는 클래스만)")
display(
    audit_class_report[audit_class_report.iloc[:, 1:].gt(0).any(axis=1)]
    .reset_index()
    .style.hide(axis="index")
)
print("문제 이미지: 전체 목록과 파일명은 CSV에 저장합니다.")
display(
    audit_problem_images.drop(columns=["file_name", "path"])
    .head(AUDIT_PREVIEW_ROWS)
    .style.hide(axis="index")
)
print(f"검사 결과: {audit_output_dir}")

## 9. 결측·겹침 요약과 원본 예시

대회 bbox 누락과 선택 bbox의 겹침 예시를 각각 최대 3장만 확인합니다. 초록은 선택 bbox, 회색은 동반 bbox, 빨강은 겹침 검수 후보입니다. 알약의 실제 존재·클래스와 bbox가 맞는지는 원본을 보고 확인해야 합니다.


In [ ]:
# 요약 그래프의 단위를 구분한다. 누락은 bbox 건수, 겹침은 bbox 쌍 수다.
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
missing_values = [
    len(audit_missing),
    int(audit_missing["대회 클래스"].sum()),
    int((~audit_missing["대회 클래스"]).sum()),
]
bars = axes[0].bar(
    ["전체", "대회 알약", "외부 전용"], missing_values, color=["#64748b", "#e76f51", "#f4a261"]
)
axes[0].bar_label(bars, padding=3)
axes[0].set(title="선택 원본 이미지의 bbox 누락", ylabel="파일명에는 있으나 라벨이 없는 알약 수")
axes[0].margins(y=0.2)
overlap_values = [
    len(audit_overlaps),
    int(audit_overlaps["선택 bbox 포함"].sum()),
    int(audit_overlaps["완전 동일 좌표"].sum()),
]
bars = axes[1].bar(
    ["전체 겹침", "선택 bbox 포함", "완전 동일 좌표"],
    overlap_values,
    color=["#64748b", "#e76f51", "#b91c1c"],
)
axes[1].bar_label(bars, padding=3)
axes[1].set(title="bbox 겹침 — 사람 검수 후보", ylabel="겹친 bbox 쌍 수")
axes[1].margins(y=0.2)
fig.tight_layout()
fig.savefig(audit_output_dir / "missing_overlap_summary.png", dpi=160, bbox_inches="tight")
plt.show()

AUDIT_EXAMPLES_PER_TYPE = 3
for audit_kind, cases in [
    ("대회 bbox 누락", audit_missing[audit_missing["대회 클래스"]]),
    ("선택 bbox 겹침", audit_overlaps[audit_overlaps["선택 bbox 포함"]]),
]:
    example_names = cases["file_name"].drop_duplicates().head(AUDIT_EXAMPLES_PER_TYPE)
    if example_names.empty:
        print(f"{audit_kind}: 해당 없음")
        continue
    fig, axes = image_grid(len(example_names), cols=3)
    for ax, file_name in zip(axes, example_names, strict=True):
        boxes = audit_valid_boxes[audit_valid_boxes["file_name"].eq(file_name)]
        missing_codes = audit_missing.loc[
            audit_missing["file_name"].eq(file_name), "category_id"
        ].tolist()
        image_pairs = audit_overlaps[audit_overlaps["file_name"].eq(file_name)]
        overlap_keys = set(
            zip(image_pairs["source_json_1"], image_pairs["annotation_id_1"], strict=True)
        )
        overlap_keys.update(
            zip(image_pairs["source_json_2"], image_pairs["annotation_id_2"], strict=True)
        )
        marked = boxes[
            [
                (source, annotation) in overlap_keys
                for source, annotation in zip(
                    boxes["source_json"], boxes["annotation_id"], strict=True
                )
            ]
        ]
        overlap_codes = sorted(
            set(image_pairs["category_id_1"]) | set(image_pairs["category_id_2"])
        )
        detail = (
            f"bbox 없는 코드: {missing_codes}"
            if audit_kind == "대회 bbox 누락"
            else f"겹침 코드: {overlap_codes}"
        )
        show_image(
            ax,
            img_path[file_name],
            title=f"이미지 {audit_numbers[file_name]} / {audit_kind}\n{detail}",
            fontsize=8,
        )
        draw_bboxes(ax, boxes, label_col="category_id", color="gray")
        draw_bboxes(ax, boxes[boxes["선택 bbox"]], label_col="category_id", color="green")
        if audit_kind == "선택 bbox 겹침":
            draw_bboxes(ax, marked, label_col="category_id", color="red")
    fig.suptitle("초록: 선택한 bbox / 회색: 동반 bbox / 빨강: 겹침 검수 후보", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    filename = "missing_examples.png" if audit_kind == "대회 bbox 누락" else "overlap_examples.png"
    fig.savefig(audit_output_dir / filename, dpi=140, bbox_inches="tight")
    plt.show()
print(
    "파일명·메타데이터 기준 검사입니다. 픽셀 속 미라벨 알약의 존재·클래스는 눈으로 확인해야 합니다."
)
print("겹침과 동일 좌표는 검수 후보이며, 이 셀은 라벨을 보정하거나 승인하지 않습니다.")

## 10. OpenCV 1차 보정 후보와 미분류 확인

§8~9에서 찾은 **157개 목표 원본 이미지의 문제만** 처리합니다. Lab 배경 분리 → 같은 조합의 정상 위치 참조 → 위치 대응이 안 되면 HSV 외형 참조 순서로 bbox 후보를 만듭니다. 클래스 연결이 모호하거나 영역이 부족하면 추정하지 않고 **미분류**로 남깁니다.

- `opencv_proposals`: 누락·겹침 보정 후보. 대회·외부 전용 클래스 모두 포함합니다.
- `opencv_unclassified`: 클래스와 영역을 연결하지 못한 항목 및 사유.
- `opencv_extra_regions`: 코드와 연결되지 않은 전경 영역. 알약인지 검수가 필요합니다.
- `opencv_image_report`: 이미지별 후보·미분류·후보에서 남은 겹침 수와 비교 그림 경로.

[AGENTS.md](../AGENTS.md)의 **“새 보정은 reviewed(사람 검수)가 있어야 적용됩니다”** 규칙에 따라 후보는 `reviewed=False`로 저장합니다. 실제 라벨·선별 데이터에는 적용하지 않습니다. `bbox_review.csv`와 이미지별 `review_*.png`에서 1차 결과를 확인합니다. 미분류와 검수 대기를 구분하며, 원본 라벨을 승인된 정답처럼 바꾸지 않습니다.


In [ ]:
import hashlib

import cv2
from PIL import Image
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

from data_process.data_processing import bbox_iou
from data_process.make_labels import (
    CORRECTION_COLS,
    assign_new_ids,
    correction_for_label,
    save_corrections,
)
from visualization import draw_bbox

CV_REPAIR_MAX_SIDE = 900
CV_REPAIR_MIN_AREA = 0.003
CV_REPAIR_MAX_AREA = 0.30
CV_REPAIR_MARGIN = 2
CV_REPAIR_MAX_DISTANCE = 0.18
CV_REPAIR_MATCH_MARGIN = 0.015
CV_REPAIR_REFERENCE_SPREAD = 0.12
CV_REPAIR_REFERENCE_MIN = 1
CV_REPAIR_SHOW_IMAGES = 3
opencv_histogram_cache = {}


def opencv_components(image_path: Path) -> tuple[np.ndarray, list[list[int]], np.ndarray]:
    """강건한 배경 조명 추정과 Lab 색 차이로 알약 몸체 영역 및 원본 좌표 후보를 추정한다."""
    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert("RGB"))
    height, width = rgb.shape[:2]
    scale = min(1.0, CV_REPAIR_MAX_SIDE / max(height, width))
    small = cv2.resize(rgb, (round(width * scale), round(height * scale)))
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32)
    # 사진의 배경 조명 경사를 추정하여 배경·그림자가 알약 영역에 합쳐지는 것을 줄인다.
    grid_y, grid_x = np.mgrid[0 : small.shape[0], 0 : small.shape[1]]
    grid_x = grid_x.astype(np.float32) / max(1, small.shape[1] - 1)
    grid_y = grid_y.astype(np.float32) / max(1, small.shape[0] - 1)
    features = np.stack(
        [
            np.ones_like(grid_x),
            grid_x,
            grid_y,
            grid_x**2,
            grid_x * grid_y,
            grid_y**2,
        ],
        axis=-1,
    )
    sample_features = features[::12, ::12].reshape(-1, 6)
    sample_colors = lab[::12, ::12].reshape(-1, 3)
    retained = np.ones(len(sample_colors), dtype=bool)
    for _iteration in range(5):
        coefficients, *_ = np.linalg.lstsq(
            sample_features[retained], sample_colors[retained], rcond=None
        )
        residuals = np.linalg.norm(sample_colors - sample_features @ coefficients, axis=1)
        retained = residuals <= np.quantile(residuals, 0.7)
    background_field = features @ coefficients
    color_delta = lab - background_field
    distance = np.clip(np.linalg.norm(color_delta, axis=2), 0, 255).astype(np.uint8)
    distance = cv2.medianBlur(distance, 5)
    otsu, _ = cv2.threshold(distance, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    median_distance = float(np.median(distance))
    deviation = float(np.median(np.abs(distance.astype(float) - median_distance)))
    threshold = max(10.0, min(float(otsu), median_distance + 4 * deviation))
    chroma_delta = np.linalg.norm(color_delta[:, :, 1:], axis=2)
    likely_body = (color_delta[:, :, 0] > 5) | (chroma_delta > 6) | (color_delta[:, :, 0] < -40)
    mask = ((distance > threshold) & likely_body).astype(np.uint8) * 255
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    kept_mask = np.zeros_like(mask)
    for contour in contours:
        area_ratio = cv2.contourArea(contour) / mask.size
        if not CV_REPAIR_MIN_AREA <= area_ratio <= CV_REPAIR_MAX_AREA:
            continue
        left, top, box_width, box_height = cv2.boundingRect(contour)
        if (
            left <= 1
            or top <= 1
            or left + box_width >= mask.shape[1] - 1
            or top + box_height >= mask.shape[0] - 1
        ):
            continue
        x0 = max(0, int(np.floor(left * width / mask.shape[1])) - CV_REPAIR_MARGIN)
        y0 = max(0, int(np.floor(top * height / mask.shape[0])) - CV_REPAIR_MARGIN)
        x1 = min(width, int(np.ceil((left + box_width) * width / mask.shape[1])) + CV_REPAIR_MARGIN)
        y1 = min(
            height, int(np.ceil((top + box_height) * height / mask.shape[0])) + CV_REPAIR_MARGIN
        )
        boxes.append([x0, y0, x1 - x0, y1 - y0])
        cv2.drawContours(kept_mask, [contour], -1, 255, thickness=-1)
    return rgb, sorted(boxes, key=lambda box: (box[0], box[1])), kept_mask


def opencv_anchor(
    file_name: str,
    category_id: int,
    reference_labels: pd.DataFrame,
    current_labels: pd.DataFrame,
    problematic_classes: set[int],
) -> tuple[np.ndarray | None, str]:
    """문제 클래스는 같은 조합·촬영 조건의 정상 라벨에서 위치 기준점을 구한다."""
    current = current_labels[current_labels["category_id"].eq(category_id)]
    if category_id not in problematic_classes and len(current) == 1:
        row = current.iloc[0]
        return np.array(
            [(row.x + row.w / 2) / row.width, (row.y + row.h / 2) / row.height]
        ), "현재 정상 라벨"
    combo = file_name.split("_")[0]
    combo_refs = reference_labels[reference_labels["file_name"].str.startswith(combo + "_")]
    basis = current.iloc[0] if not current.empty else current_labels.iloc[0]
    target_refs = combo_refs[combo_refs["category_id"].eq(category_id)]
    for field in ["camera_lo", "drug_dir", "size"]:
        if field in target_refs and pd.notna(basis.get(field)):
            target_refs = target_refs[target_refs[field].eq(basis[field])]
    trusted = current_labels[~current_labels["category_id"].isin(problematic_classes)]
    trusted = trusted.drop_duplicates("category_id").set_index("category_id")
    predicted_centers = []
    for ref_name in target_refs["file_name"].unique():
        reference = combo_refs[combo_refs["file_name"].eq(ref_name)].set_index("category_id")
        anchors = sorted(set(reference.index) & set(trusted.index))
        if len(anchors) < 2:
            continue
        reference_anchor = reference.loc[anchors]
        current_anchor = trusted.loc[anchors]
        before = np.column_stack(
            [
                (reference_anchor.x + reference_anchor.w / 2) / reference_anchor.width,
                (reference_anchor.y + reference_anchor.h / 2) / reference_anchor.height,
            ]
        )
        after = np.column_stack(
            [
                (current_anchor.x + current_anchor.w / 2) / current_anchor.width,
                (current_anchor.y + current_anchor.h / 2) / current_anchor.height,
            ]
        )
        # 정상 약 2개 이상으로 회전·크기 변화·이동을 추정해 촬영 각도 차이를 정합한다.
        matrix = np.zeros((2 * len(anchors), 4))
        matrix[0::2] = np.column_stack(
            [before[:, 0], -before[:, 1], np.ones(len(anchors)), np.zeros(len(anchors))]
        )
        matrix[1::2] = np.column_stack(
            [before[:, 1], before[:, 0], np.zeros(len(anchors)), np.ones(len(anchors))]
        )
        if np.linalg.matrix_rank(matrix) < 4:
            continue
        transform, *_ = np.linalg.lstsq(matrix, after.ravel(), rcond=None)
        scale = np.hypot(transform[0], transform[1])
        fit_error = np.linalg.norm((matrix @ transform).reshape(-1, 2) - after, axis=1).max()
        if not 0.5 <= scale <= 2.0 or fit_error > 0.03:
            continue
        target = reference.loc[category_id]
        center = np.array(
            [(target.x + target.w / 2) / target.width, (target.y + target.h / 2) / target.height]
        )
        predicted = np.array(
            [
                transform[0] * center[0] - transform[1] * center[1] + transform[2],
                transform[1] * center[0] + transform[0] * center[1] + transform[3],
            ]
        )
        if ((predicted >= 0) & (predicted <= 1)).all():
            predicted_centers.append(predicted)
    if len(predicted_centers) < CV_REPAIR_REFERENCE_MIN:
        return None, "정상 약 기준 위치 정합 참조 부족"
    centers = np.asarray(predicted_centers)
    center = np.median(centers, axis=0)
    if np.quantile(np.linalg.norm(centers - center, axis=1), 0.9) > CV_REPAIR_REFERENCE_SPREAD:
        return None, "위치 정합 후 참조 좌표가 불안정"
    return center, f"같은 조합 정상 이미지 {len(centers)}장 위치 정합"


def opencv_color_histogram(image_path: Path, bbox: list[float]) -> np.ndarray:
    """다른 조합의 외형을 참고하기 위해 bbox 중앙부 HSV 분포를 계산한다."""
    cache_key = (str(image_path), tuple(int(round(value)) for value in bbox))
    if cache_key in opencv_histogram_cache:
        return opencv_histogram_cache[cache_key]
    with Image.open(image_path) as image:
        left = max(0, int(bbox[0]))
        top = max(0, int(bbox[1]))
        right = min(image.width, int(bbox[0] + bbox[2]))
        bottom = min(image.height, int(bbox[1] + bbox[3]))
        if right <= left or bottom <= top:
            raise ValueError("외형 참조 bbox가 비어 있습니다.")
        crop = np.asarray(image.convert("RGB").crop((left, top, right, bottom)))
    height, width = crop.shape[:2]
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.ellipse(
        mask,
        (width // 2, height // 2),
        (max(1, int(width * 0.28)), max(1, int(height * 0.28))),
        0,
        0,
        360,
        255,
        -1,
    )
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    histogram = cv2.calcHist([hsv], [0, 1, 2], mask, [12, 8, 4], [0, 180, 0, 256, 0, 256])
    histogram /= max(float(histogram.sum()), 1.0)
    histogram = histogram.astype(np.float32).reshape(-1, 1)
    opencv_histogram_cache[cache_key] = histogram
    return histogram


def opencv_appearance_assign(
    file_name: str,
    boxes: list[list[int]],
    reference_labels: pd.DataFrame,
) -> tuple[dict[int, list[int]], str]:
    """위치 참조가 없으면 다른 조합의 색·형태가 뚜렷이 구분될 때만 외형 후보를 낸다."""
    codes = sorted({int(code) for code in file_name.split("_")[0].split("-")[1:]})
    if len(boxes) < len(codes):
        return {}, "HSV 외형 참고 보류: 파일명 알약 영역 부족"
    candidate_histograms = [opencv_color_histogram(img_path[file_name], box) for box in boxes]
    costs = np.zeros((len(codes), len(boxes)))
    for position, code in enumerate(codes):
        references = reference_labels[reference_labels["category_id"].eq(code)].copy()
        references["combo"] = references["file_name"].str.split("_").str[0]
        references = references.sort_values("file_name").drop_duplicates("combo").head(3)
        if len(references) < 2:
            return {}, f"HSV 외형 참고 보류: 클래스 {code}의 다른 조합 참조 부족"
        for component, box in enumerate(boxes):
            scores = []
            for row in references.itertuples():
                reference_box = [row.x, row.y, row.w, row.h]
                histogram = opencv_color_histogram(img_path[row.file_name], reference_box)
                color_distance = cv2.compareHist(
                    histogram, candidate_histograms[component], cv2.HISTCMP_BHATTACHARYYA
                )
                shape_ratio = max(box[2:]) / min(box[2:])
                reference_ratio = max(row.w, row.h) / min(row.w, row.h)
                scores.append(color_distance + 0.15 * abs(np.log(shape_ratio / reference_ratio)))
            costs[position, component] = float(np.median(scores))
    rows, columns = linear_sum_assignment(costs)
    if costs[rows, columns].max() > 0.65:
        return {}, "HSV 외형 참고 보류: 색·형태가 정상 참조와 다름"
    best_cost = costs[rows, columns].sum()
    for row, column in zip(rows, columns, strict=True):
        alternative = costs.copy()
        alternative[row, column] = 1e6
        alt_rows, alt_columns = linear_sum_assignment(alternative)
        if alternative[alt_rows, alt_columns].sum() - best_cost < 0.10:
            return {}, "HSV 외형 참고 보류: 클래스 간 외형 구분이 모호함"
    return {
        codes[row]: boxes[column] for row, column in zip(rows, columns, strict=True)
    }, "HSV 외형 참고 후보: 다른 조합 색·형태 비교 (사람 검수 필요)"


def opencv_assign(
    file_name: str,
    boxes: list[list[int]],
    rgb_shape: tuple,
    reference_labels: pd.DataFrame,
    current_labels: pd.DataFrame,
    problematic_classes: set[int],
) -> tuple[dict[int, list[int]], str]:
    """전체 약 코드와 영역을 일대일 연결하되 개수·거리·대안 모호성을 검사한다."""
    codes = sorted({int(code) for code in file_name.split("_")[0].split("-")[1:]})
    if len(boxes) < len(codes):
        return {}, f"영역 {len(boxes)}개 < 파일명 알약 {len(codes)}개"
    # 다른 라벨과 중복되지 않고 몸체에 잘 맞는 현재 bbox는 위치 정합 기준으로 쓴다.
    overlap_scores = np.array(
        [
            [bbox_iou(row[["x", "y", "w", "h"]].to_numpy(dtype=float), box) for box in boxes]
            for _, row in current_labels.iterrows()
        ]
    )
    reliable_current = set()
    for position, row in enumerate(current_labels.itertuples()):
        best_component = int(overlap_scores[position].argmax())
        strong_matches = overlap_scores[:, best_component] >= 0.72
        if strong_matches[position] and strong_matches.sum() == 1:
            reliable_current.add(int(row.category_id))
    problematic_classes = set(problematic_classes) - reliable_current
    anchors = []
    sources = []
    for code in codes:
        anchor, source = opencv_anchor(
            file_name,
            code,
            reference_labels,
            current_labels,
            problematic_classes,
        )
        if anchor is None:
            return opencv_appearance_assign(file_name, boxes, reference_labels)
        anchors.append(anchor)
        sources.append(source)
    height, width = rgb_shape[:2]
    centers = np.array([[(x + w / 2) / width, (y + h / 2) / height] for x, y, w, h in boxes])
    costs = np.linalg.norm(np.asarray(anchors)[:, None, :] - centers[None, :, :], axis=2)
    rows, columns = linear_sum_assignment(costs)
    if costs[rows, columns].max() > CV_REPAIR_MAX_DISTANCE:
        return {}, "참조 위치와 분리한 알약의 거리가 큼"
    best_cost = costs[rows, columns].sum()
    for row, column in zip(rows, columns, strict=True):
        alternative = costs.copy()
        alternative[row, column] = 1e6
        alt_rows, alt_columns = linear_sum_assignment(alternative)
        if alternative[alt_rows, alt_columns].sum() - best_cost < CV_REPAIR_MATCH_MARGIN:
            return {}, "클래스-영역 대응이 모호함"
    return {
        codes[row]: boxes[column] for row, column in zip(rows, columns, strict=True)
    }, "; ".join(sorted(set(sources)))

In [ ]:
if any(
    name not in globals()
    for name in ["audit_problem_images", "audit_annotations", "audit_missing", "audit_overlaps"]
):
    raise RuntimeError("먼저 §8~9의 157개 목표 데이터 결측 검사를 실행하세요.")
cv2.setNumThreads(1)
# 실제 사용하기로 한 원본 이미지 5,499장 중 문제 95장만 픽셀을 읽는다.
opencv_issue_files = sorted(audit_problem_images["file_name"].unique())
opencv_input_signature = (
    audit_annotations[
        ["file_name", "category_id", "source_json", "annotation_id", "x", "y", "w", "h"]
    ]
    .sort_values(["file_name", "category_id"])
    .to_csv(index=False)
)
opencv_fingerprint = hashlib.sha256(opencv_input_signature.encode()).hexdigest()[:16]
opencv_output_dir = ROOT / "runs/original157_opencv_review" / opencv_fingerprint
opencv_output_dir.mkdir(parents=True, exist_ok=True)
opencv_review_path = opencv_output_dir / "bbox_review.csv"
# 문제 이미지와 잘못된 좌표·강한 겹침이 있는 원본은 정상 위치·외형 참고에서 제외한다.
opencv_all_pairs = find_box_pairs(available_ann_df, max_gap=0)
opencv_bad_reference_files = set(invalid_files) | set(opencv_issue_files)
opencv_bad_reference_files |= set(
    opencv_all_pairs.loc[opencv_all_pairs["iou"].ge(0.5), "file_name"]
)
opencv_label_codes = available_ann_df.groupby("file_name")["category_id"].agg(set)
opencv_complete_reference_names = {
    name
    for name, codes in opencv_label_codes.items()
    if codes == {int(value) for value in name.split("_")[0].split("-")[1:]}
}
opencv_reference_labels = available_ann_df[
    ~available_ann_df["file_name"].isin(opencv_bad_reference_files)
    & available_ann_df["file_name"].isin(opencv_complete_reference_names)
].copy()
opencv_proposal_rows = []
opencv_component_rows = []
opencv_image_rows = []
opencv_extra_region_rows = []
opencv_cases = {}
for file_number, file_name in enumerate(
    tqdm(opencv_issue_files, desc="OpenCV 누락·겹침 후보 분석"), 1
):
    current = audit_annotations[audit_annotations["file_name"].eq(file_name)].copy()
    missing_codes = set(audit_missing.loc[audit_missing["file_name"].eq(file_name), "category_id"])
    pairs = audit_overlaps[audit_overlaps["file_name"].eq(file_name)]
    problem_codes = missing_codes | set(pairs["category_id_1"]) | set(pairs["category_id_2"])
    invalid_codes = set(audit_invalid.loc[audit_invalid["file_name"].eq(file_name), "category_id"])
    problem_codes |= invalid_codes
    expected_codes = {int(value) for value in file_name.split("_")[0].split("-")[1:]}
    assignment = {}
    components = []
    source = "미분류: 분석 전"
    preview_path = opencv_output_dir / f"review_{file_number:03d}.png"
    try:
        rgb, components, mask = opencv_components(img_path[file_name])
        if len(components) < len(expected_codes):
            # 작은 알약을 놓친 경우에만 최소 면적을 낮춰 한 번 더 확인한다.
            saved_minimum = CV_REPAIR_MIN_AREA
            CV_REPAIR_MIN_AREA = 0.001
            try:
                rgb_retry, components_retry, mask_retry = opencv_components(img_path[file_name])
            finally:
                CV_REPAIR_MIN_AREA = saved_minimum
            if len(components_retry) >= len(components):
                rgb, components, mask = rgb_retry, components_retry, mask_retry
        assignment, source = opencv_assign(
            file_name,
            components,
            rgb.shape,
            opencv_reference_labels,
            current,
            problem_codes,
        )
        if not assignment and len(components) >= len(expected_codes):
            appearance, appearance_source = opencv_appearance_assign(
                file_name, components, opencv_reference_labels
            )
            if appearance:
                assignment, source = appearance, appearance_source
            else:
                source = f"{source}; {appearance_source}"
        # 모든 코드가 한 번씩 연결되고 같은 영역을 중복 사용하지 않았을 때만 후보를 유지한다.
        if assignment and (
            set(assignment) != expected_codes
            or len({tuple(box) for box in assignment.values()}) != len(expected_codes)
        ):
            assignment, source = {}, "미분류: 전체 클래스와 영역의 일대일 대응 실패"
        for component_number, box in enumerate(components, 1):
            opencv_component_rows.append(
                {
                    "file_name": file_name,
                    "이미지 번호": int(audit_numbers[file_name]),
                    "영역 번호": component_number,
                    "x": box[0],
                    "y": box[1],
                    "w": box[2],
                    "h": box[3],
                }
            )
        used_boxes = {tuple(box) for box in assignment.values()}
        for component_number, box in enumerate(components, 1):
            if assignment and tuple(box) not in used_boxes:
                opencv_extra_region_rows.append(
                    {
                        "file_name": file_name,
                        "이미지 번호": int(audit_numbers[file_name]),
                        "영역 번호": component_number,
                        "bbox": box,
                        "상태": "미분류 전경: 알약 여부 검수 필요",
                        "검수 그림": str(preview_path),
                    }
                )
        # 전후 그림: 오른쪽은 미검수 후보이며 실제 라벨 적용 결과가 아니다.
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax in axes:
            ax.imshow(rgb)
            ax.axis("off")
        for row in current.itertuples():
            draw_bbox(
                axes[0],
                [row.x, row.y, row.w, row.h],
                label=str(int(row.category_id)),
                color="lime" if row._asdict().get("선택 bbox", False) else "gray",
            )
        for component_number, box in enumerate(components, 1):
            draw_bbox(axes[1], box, label=f"영역 {component_number}", color="cyan")
        for row in current.itertuples():
            if int(row.category_id) not in problem_codes:
                draw_bbox(
                    axes[2],
                    [row.x, row.y, row.w, row.h],
                    label=str(int(row.category_id)),
                    color="gray",
                )
        for code in sorted(problem_codes):
            if code in assignment:
                draw_bbox(axes[2], assignment[code], label=f"{code} 후보", color="lime")
        axes[0].set_title("기존 라벨", fontsize=10)
        axes[1].set_title(
            f"OpenCV 전경 {len(components)}개 / 파일명 약 {len(expected_codes)}개", fontsize=10
        )
        axes[2].set_title(
            "미검수 보정 후보" if assignment else "미분류·위치 연결 보류", fontsize=10
        )
        fig.suptitle(f"이미지 {audit_numbers[file_name]} / {source}", fontsize=8)
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        fig.savefig(preview_path, dpi=100, bbox_inches="tight")
        plt.close(fig)
    except (OSError, ValueError, cv2.error, KeyError) as error:
        assignment, source = {}, f"미분류: {type(error).__name__}: {error}"
        plt.close("all")
    # 누락 bbox와 겹침에 관여한 모든 알약에 대해 한 행씩 남긴다. 대회·외부 모두 포함한다.
    for code in sorted(problem_codes):
        original_rows = current[current["category_id"].eq(code)]
        candidate_box = assignment.get(code)
        if len(original_rows) == 1:
            original = original_rows.iloc[0]
            record = correction_for_label(
                audit_annotations,
                original["source_json"],
                int(original["annotation_id"]),
                candidate_box
                if candidate_box is not None
                else original[["x", "y", "w", "h"]].tolist(),
                EXTRACT_DIR,
                correction_id=f"original157_cv:{file_name}:{code}",
                stage="위치 수정",
                case="bbox 겹침",
                method=source,
                author="OpenCV 후보",
                reviewed=False,
                source_record=str(preview_path),
            )
            if candidate_box is None:
                record["new_bbox"] = None
        else:
            record = dict.fromkeys(CORRECTION_COLS)
            record.update(
                correction_id=f"original157_cv:{file_name}:{code}",
                stage="추가",
                case="bbox 누락",
                file_name=file_name,
                category_id=int(code),
                new_bbox=candidate_box,
                method=source,
                author="OpenCV 후보",
                reviewed=False,
                source_record=str(preview_path),
                map_status="새 라벨",
            )
        record.update(
            auto_status="보정 후보" if candidate_box is not None else "미분류",
            reason=source,
            decision="",
            review_png=str(preview_path),
            competition_class=int(code) in audit_competition_ids,
        )
        opencv_proposal_rows.append(record)
    # 후보 좌표끼리의 잔여 겹침만 계산한다. 원본·실제 라벨에는 적용하지 않는다.
    candidate_boxes = []
    candidate_codes = []
    for code in sorted(expected_codes):
        old = current[current["category_id"].eq(code)]
        box = (
            assignment.get(code)
            if code in problem_codes
            else (old.iloc[0][["x", "y", "w", "h"]].tolist() if len(old) == 1 else None)
        )
        if box is not None:
            candidate_codes.append(code)
            candidate_boxes.append(box)
    remaining_pairs = []
    for left in range(len(candidate_boxes)):
        for right in range(left + 1, len(candidate_boxes)):
            if bbox_iou(candidate_boxes[left], candidate_boxes[right]) > 0:
                remaining_pairs.append([candidate_codes[left], candidate_codes[right]])
    unassigned_codes = sorted(problem_codes - set(assignment))
    opencv_image_rows.append(
        {
            "file_name": file_name,
            "이미지 번호": int(audit_numbers[file_name]),
            "문제 알약 수": len(problem_codes),
            "보정 후보 수": len(problem_codes) - len(unassigned_codes),
            "미분류 수": len(unassigned_codes),
            "미분류 코드": str(unassigned_codes),
            "후보에서 남은 겹침 쌍": len(remaining_pairs),
            "잔여 겹침 코드": str(remaining_pairs),
            "상태": "미분류"
            if unassigned_codes
            else ("후보 겹침 검수" if remaining_pairs else "후보 생성·검수 대기"),
            "이유": source,
            "검수 그림": str(preview_path),
        }
    )
    opencv_cases[file_name] = {
        "components": components,
        "assignment": assignment,
        "reason": source,
        "unclassified": unassigned_codes,
        "preview": str(preview_path),
    }
opencv_proposals = assign_new_ids(pd.DataFrame(opencv_proposal_rows), audit_annotations)
# 같은 입력으로 재실행해도 사람이 입력한 결정과 좌표를 덮어쓰지 않는다.
if opencv_review_path.is_file():
    existing_review = pd.read_csv(opencv_review_path, keep_default_na=False)
    if existing_review["correction_id"].duplicated().any():
        raise ValueError("기존 검수 CSV에 correction_id 중복이 있습니다.")
    existing_review = existing_review.set_index("correction_id")
    for row_number, row in opencv_proposals.iterrows():
        if row["correction_id"] in existing_review.index:
            previous = existing_review.loc[row["correction_id"]]
            current_hash = row.get("input_sha256")
            current_hash = "" if pd.isna(current_hash) else str(current_hash)
            if str(previous.get("input_sha256", "")) != current_hash:
                raise ValueError(
                    "원본 JSON fingerprint가 달라 기존 검수 결정을 재사용하지 않습니다."
                )
            if str(previous.get("decision", "")).strip():
                for column in ["decision", "new_bbox"]:
                    opencv_proposals.at[row_number, column] = previous[column]
save_corrections(opencv_proposals, opencv_review_path)
opencv_image_report = pd.DataFrame(opencv_image_rows)
opencv_unclassified = opencv_proposals[opencv_proposals["auto_status"].eq("미분류")].copy()
opencv_extra_regions = pd.DataFrame(
    opencv_extra_region_rows,
    columns=["file_name", "이미지 번호", "영역 번호", "bbox", "상태", "검수 그림"],
)
opencv_image_report.to_csv(
    opencv_output_dir / "image_status.csv", index=False, encoding="utf-8-sig"
)
opencv_unclassified.to_csv(
    opencv_output_dir / "unclassified.csv", index=False, encoding="utf-8-sig"
)
opencv_extra_regions.to_csv(
    opencv_output_dir / "unclassified_regions.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(opencv_component_rows).to_csv(
    opencv_output_dir / "opencv_regions.csv", index=False, encoding="utf-8-sig"
)
print(f"전체 문제 이미지 {len(opencv_issue_files)}장 처리")
print(
    f"보정 후보 {(opencv_proposals['auto_status'] == '보정 후보').sum()}건 / "
    f"미분류 {len(opencv_unclassified)}건"
)
print(
    "후보 좌표에서 겹침이 남은 이미지:",
    int(opencv_image_report["후보에서 남은 겹침 쌍"].gt(0).sum()),
    "장",
)
print("실제 보정 적용 0건: 후보는 모두 사람 검수 전입니다.")
display(
    opencv_image_report[
        opencv_image_report["미분류 수"].gt(0) | opencv_image_report["후보에서 남은 겹침 쌍"].gt(0)
    ].drop(columns=["file_name", "검수 그림"])
)
print("미분류 항목: 클래스 번호 / 이유 / 그림 경로")
display(opencv_unclassified[["file_name", "category_id", "reason", "review_png"]])
if not opencv_extra_regions.empty:
    print("추가 전경 영역: 알약인지 확인하지 못한 미분류 영역")
    display(opencv_extra_regions)
print(f"검수 CSV: {opencv_review_path}")

## 11. 검수한 보정 적용 → 남은 미분류·누락·겹침 재검사

§10의 `bbox_review.csv`를 확인하고 이 셀을 실행합니다.

- `decision=승인`: `new_bbox` 좌표를 적용합니다. 미분류도 사람이 `[x,y,w,h]`를 적고 승인하면 처리할 수 있습니다.
- `decision=원본 유지`: 기존 bbox가 맞다는 확인입니다. 누락 라벨의 추가 행에는 사용할 수 없습니다.
- 빈 값·`거부`: 적용하지 않고 보류합니다. `reviewed` 열만 바꿔도 승인되지 않습니다.

원본 JSON fingerprint와 bbox 범위·대상 식별자를 검사한 뒤 기존 `apply_corrections()`로 **사람이 승인한 행만** 메모리 복사본에 적용합니다. 원본 이미지·JSON·파이프라인 라벨은 바꾸지 않습니다.

- `opencv_applied_annotations`: 승인된 보정을 반영한 별도 라벨 표.
- `opencv_selected_pills_after`: 처음 선택한 bbox의 승인 후 좌표.
- `opencv_ready_images`, `opencv_ready_annotations`: 미검수·미분류·미해결 누락/겹침이 없는 이미지·라벨.
- `opencv_held_images`: 보류 이미지. CSV에서 확인할 수 있습니다.
- `opencv_remaining_unclassified`, `opencv_remaining_regions`: 남은 미분류와 추가 전경 영역.

겹침 양쪽을 모두 `원본 유지`로 검수한 경우에만 정상 확인된 겹침으로 구분합니다. 추가 전경 영역은 알약 여부가 확인되기 전까지 보류합니다. 학습용 txt·data.yaml·분할 목록은 만들지 않습니다.


In [ ]:
import ast

from data_process.make_labels import apply_corrections, file_sha256


def opencv_parse_bbox(value: object) -> list[int] | None:
    """CSV의 bbox를 안전하게 읽고 유한한 4개 좌표인지 확인한다."""
    if isinstance(value, str):
        if not value.strip():
            return None
        value = ast.literal_eval(value)
    if value is None or not isinstance(value, (list, tuple)):
        return None
    if len(value) != 4 or any(isinstance(number, bool) for number in value):
        return None
    numbers = np.asarray(value, dtype=float)
    if not np.isfinite(numbers).all():
        return None
    return np.rint(numbers).astype(int).tolist()


def opencv_apply_review(
    original: pd.DataFrame,
    expected: pd.DataFrame,
    review: pd.DataFrame,
    annotation_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """사람이 결정한 승인·원본 유지만 처리하고 검수 전 기록은 보류한다."""
    if review["correction_id"].duplicated().any() or expected["correction_id"].duplicated().any():
        raise ValueError("correction_id가 중복되어 적용을 중단합니다.")
    if set(review["correction_id"]) != set(expected["correction_id"]):
        raise ValueError("현재 후보와 검수 CSV의 correction_id 구성이 다릅니다.")
    review = review.set_index("correction_id")
    approved = []
    logs = []
    for _, record in expected.iterrows():
        record = record.copy()
        edited = review.loc[record["correction_id"]]
        decision = str(edited.get("decision", "")).strip()
        log = {
            "correction_id": record["correction_id"],
            "file_name": record["file_name"],
            "category_id": int(record["category_id"]),
            "decision": decision,
        }
        if decision not in {"승인", "원본 유지"}:
            logs.append(
                {**log, "status": "보류", "reasons": "거부" if decision == "거부" else "미검수"}
            )
            continue
        # CSV에서 대상 라벨의 식별자를 바꿔 다른 라벨에 적용하는 실수를 방지한다.
        changed_identity = any(
            str(edited.get(column)) != str(record[column]) for column in ["file_name", "stage"]
        )
        changed_identity |= pd.isna(edited.get("category_id")) or int(edited["category_id"]) != int(
            record["category_id"]
        )
        if changed_identity:
            logs.append({**log, "status": "거부", "reasons": "대상 파일·단계·클래스 변경"})
            continue
        source = record.get("source_json")
        if pd.notna(source):
            source_path = annotation_dir / str(source)
            if not source_path.is_file() or file_sha256(source_path) != record.get("input_sha256"):
                logs.append({**log, "status": "거부", "reasons": "원본 JSON fingerprint 불일치"})
                continue
        if decision == "원본 유지":
            if record["stage"] == "추가":
                logs.append({**log, "status": "거부", "reasons": "누락 bbox에는 원본 유지 불가"})
            else:
                logs.append({**log, "status": "원본 유지", "reasons": "사람이 기존 bbox 확인"})
            continue
        try:
            new_bbox = opencv_parse_bbox(edited.get("new_bbox"))
        except (ValueError, SyntaxError, TypeError, OverflowError):
            new_bbox = None
        if new_bbox is None:
            logs.append({**log, "status": "거부", "reasons": "new_bbox에 [x,y,w,h] 좌표 필요"})
            continue
        if record["stage"] != "추가":
            record["old_bbox"] = opencv_parse_bbox(record["old_bbox"])
        record["new_bbox"] = new_bbox
        # decision='승인'이라는 사람의 명시적 결정만 reviewed로 변환한다.
        record["reviewed"] = True
        approved.append(record)
    if approved:
        corrected, application_log = apply_corrections(original, pd.DataFrame(approved))
        for row in application_log.to_dict("records"):
            row["decision"] = "승인"
            logs.append(row)
    else:
        corrected = original.copy()
    return corrected, pd.DataFrame(
        logs, columns=["correction_id", "file_name", "category_id", "decision", "status", "reasons"]
    )


if any(
    name not in globals()
    for name in ["opencv_review_path", "opencv_proposals", "audit_annotations"]
):
    raise RuntimeError("먼저 §10의 OpenCV 1차 분석을 실행하세요.")
opencv_human_review = pd.read_csv(opencv_review_path, keep_default_na=False)
opencv_applied_annotations, opencv_application_log = opencv_apply_review(
    audit_annotations,
    opencv_proposals,
    opencv_human_review,
    EXTRACT_DIR,
)
opencv_resolved_ids = set(
    opencv_application_log.loc[
        opencv_application_log["status"].isin(["적용", "이미 적용", "원본 유지"]), "correction_id"
    ]
)
opencv_pending_review = opencv_proposals[
    ~opencv_proposals["correction_id"].isin(opencv_resolved_ids)
].copy()
opencv_remaining_unclassified = opencv_pending_review[
    opencv_pending_review["auto_status"].eq("미분류")
].copy()

# 적용 후 원본 이미지 전체의 누락·겹침을 다시 확인한다.
opencv_post_missing = audit_expected.merge(
    opencv_applied_annotations[["file_name", "category_id"]].drop_duplicates(),
    on=["file_name", "category_id"],
    how="left",
    indicator=True,
    validate="one_to_one",
)
opencv_post_missing = opencv_post_missing[opencv_post_missing["_merge"].eq("left_only")].drop(
    columns="_merge"
)
opencv_post_pair_rows = find_box_pairs(opencv_applied_annotations, max_gap=0)
opencv_post_overlaps = opencv_post_pair_rows[opencv_post_pair_rows["iou"].gt(0)].copy()
# 양쪽 bbox 모두 '원본 유지'로 확인한 쌍은 사람이 정상으로 확인한 겹침으로 구분한다.
opencv_keep_ids = set(
    opencv_application_log.loc[opencv_application_log["status"].eq("원본 유지"), "correction_id"]
)
opencv_keep_records = opencv_proposals[opencv_proposals["correction_id"].isin(opencv_keep_ids)]
opencv_keep_keys = set(
    zip(opencv_keep_records["source_json"], opencv_keep_records["annotation_id"], strict=True)
)
opencv_post_overlaps["사람 확인 겹침"] = [
    (row.source_json_1, row.annotation_id_1) in opencv_keep_keys
    and (row.source_json_2, row.annotation_id_2) in opencv_keep_keys
    for row in opencv_post_overlaps.itertuples()
]
opencv_post_invalid_flags = bbox_flags(opencv_applied_annotations)
opencv_post_invalid = opencv_applied_annotations[
    ~np.isfinite(opencv_applied_annotations[["x", "y", "w", "h"]].to_numpy(dtype=float)).all(axis=1)
    | opencv_post_invalid_flags["out_of_image"]
    | opencv_post_invalid_flags["non_positive"]
    | opencv_applied_annotations.duplicated(["file_name", "category_id"], keep=False)
].copy()
opencv_remaining_regions = opencv_extra_regions.copy()
opencv_held_names = set(opencv_pending_review["file_name"]) | set(opencv_post_missing["file_name"])
opencv_held_names |= set(
    opencv_post_overlaps.loc[~opencv_post_overlaps["사람 확인 겹침"], "file_name"]
)
opencv_held_names |= set(opencv_post_invalid["file_name"]) | set(
    opencv_remaining_regions["file_name"]
)
opencv_ready_images = audit_image_lookup[
    ~audit_image_lookup["file_name"].isin(opencv_held_names)
].copy()
opencv_held_images = audit_image_lookup[
    audit_image_lookup["file_name"].isin(opencv_held_names)
].copy()
opencv_ready_annotations = opencv_applied_annotations[
    opencv_applied_annotations["file_name"].isin(opencv_ready_images["file_name"])
].copy()
# 선택한 7,803개 bbox의 식별자는 그대로 두고 승인된 좌표만 반영한 별도 표다.
opencv_selected_pills_after = opencv_applied_annotations[
    [
        (source, annotation) in audit_selected_keys
        for source, annotation in zip(
            opencv_applied_annotations["source_json"],
            opencv_applied_annotations["annotation_id"],
            strict=True,
        )
    ]
].copy()
for name, frame in {
    "application_log": opencv_application_log,
    "approved_annotations": opencv_applied_annotations,
    "remaining_unclassified": opencv_remaining_unclassified,
    "remaining_missing": opencv_post_missing,
    "remaining_overlaps": opencv_post_overlaps,
    "remaining_invalid": opencv_post_invalid,
    "remaining_regions": opencv_remaining_regions,
    "ready_images": opencv_ready_images,
    "held_images": opencv_held_images,
}.items():
    frame.to_csv(opencv_output_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
print("적용 결과:", opencv_application_log["status"].value_counts().to_dict())
print(
    f"남은 미분류 {len(opencv_remaining_unclassified)}건 / bbox 누락 {len(opencv_post_missing)}건"
)
print(
    f"미확인 겹침 {int((~opencv_post_overlaps['사람 확인 겹침']).sum())}쌍 / "
    f"추가 미분류 전경 {len(opencv_remaining_regions)}개"
)
print(f"검사 통과 이미지 {len(opencv_ready_images):,}장 / 보류 {len(opencv_held_images):,}장")
display(opencv_application_log[~opencv_application_log["status"].isin(["적용", "원본 유지"])])
display(opencv_remaining_unclassified[["file_name", "category_id", "reason", "review_png"]])
print("적용 결과는 메모리와 runs의 별도 CSV입니다. 원본·기존 파이프라인 라벨은 수정하지 않습니다.")

## 12. 원본 이미지의 대회 알약 bbox 누락 15건 시각화

파일명에는 대회 알약 클래스가 있지만 annotation에는 bbox가 없는 15건을 전부 표시합니다. 기존 bbox는 회색으로 그리고, 누락 클래스 번호는 제목에 표시합니다.

In [ ]:
required_names = ["audit_missing", "audit_valid_boxes", "audit_numbers", "img_path"]
if any(name not in globals() for name in required_names):
    raise RuntimeError("먼저 §8의 원본 이미지 결측치 검사를 실행하세요.")

competition_missing = (
    audit_missing[audit_missing["대회 클래스"]]
    .sort_values(["이미지 번호", "category_id"])
    .reset_index(drop=True)
)
if competition_missing.empty:
    print("대회 알약 bbox 누락이 없습니다.")
else:
    cols = 3
    rows = int(np.ceil(len(competition_missing) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, missing) in zip(axes, competition_missing.iterrows(), strict=False):
        boxes = audit_valid_boxes[audit_valid_boxes["file_name"].eq(missing["file_name"])]
        show_image(
            ax,
            img_path[missing["file_name"]],
            title=(
                f"이미지 {missing['이미지 번호']} | 누락 클래스 {missing['category_id']}\n"
                f"기존 bbox {len(boxes)}개"
            ),
            fontsize=9,
        )
        draw_bboxes(ax, boxes, label_col="category_id", color="gray")
    for ax in axes[len(competition_missing) :]:
        ax.axis("off")
    fig.suptitle(
        f"원본 이미지의 대회 알약 bbox 누락 {len(competition_missing)}건",
        fontsize=15,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    output_path = audit_output_dir / "competition_missing_15_all.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    display(
        competition_missing[["이미지 번호", "category_id"]]
        .rename(columns={"category_id": "누락 클래스"})
        .style.hide(axis="index")
    )
    print(f"저장: {output_path}")